In [1]:
%matplotlib inline
# Cell 1 — WO-02 Step 3, family 1: Position. Load fixtures and population payloads (read-only; the API is never called)
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.edop.dashboard import wo02_payloads as P, wo02_rules_position as R, wo02_analysis as A

fx, pop, prov = P.load_all()
W = P.population_weights(pop)
FX_BY_ID = {q['id']: q for q in fx}
print('provenance:', prov)
print(f'fixtures with a 200: {len(fx)} | population queries with a 200: {len(pop)}')
print('weights (each kind sums to 1):', {k: round(sum(v for i, v in W.items() if i.endswith(k)), 3) for k in 'sbt'})


provenance: {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
fixtures with a 200: 33 | population queries with a 200: 505
weights (each kind sums to 1): {'s': 1.0, 'b': 1.0, 't': 1.0}


In [6]:
# Cell 2 — The seven Position statement types (the rules live in scripts/edop/dashboard/wo02_rules_position.py)
for s in R.STATEMENTS:
    print(f"{s['id']}  {s['name']}")
    print(f"      applies to: {s['applies_to']}")
    print(f"      template  : {s['template']}")
print()
print(f"zero-aware threshold (engine's own): {R.ZERO_AWARE_THRESHOLD} | POS-06 display floor: {R.FLOOR}")


POS-01  Percentile with reference frame (one basin)
      applies to: single basin and one-basin buffer; area_weighted, distribution_only, dominant_basin; L6, L8
      template  : {name} ranks at the {ord} percentile among all L{level} basins worldwide.
POS-02  Percentile for an area (area-weighted mean)
      applies to: buffer, polity; area_weighted, distribution_only; L6, L8
      template  : Across the {n} basins in this {area}, {name} averages the {ord} percentile among all L{level} basins worldwide (area-weighted mean of the basins' percentiles).
POS-03  Raw value with units (one basin)
      applies to: single basin and one-basin buffer; area_weighted, distribution_only, dominant_basin
      template  : {name} is {raw}{ units}.
POS-04  Zero here (one basin)
      applies to: single basin and one-basin buffer; area_weighted, distribution_only, dominant_basin
      template  : {name} is zero in this basin.
POS-05  Rank among the non-zero basins (zero-aware, one basin)
      applie

In [2]:
# Cell 3 — Fixtures: apply every rule to every Band A-E row of the fixtures, and say why not where it does not apply
fdf = A.apply_fixtures(fx, R)
print('rows evaluated:', len(fdf) // len(R.STATEMENTS), '(Band A-E rows of', len(fx), 'fixtures)')
print(A.fixture_table(fdf).to_string())
print()
print('why a statement did not apply (count of rows):')
print(A.reason_table(fdf).to_string())


rows evaluated: 1664 (Band A-E rows of 33 fixtures)
res                  applied  not_applicable
id     scope                                
POS-01 buffer              0             260
       polity              0             208
       single_basin      651             545
POS-02 buffer             96             164
       polity            146              62
       single_basin        0            1196
POS-03 buffer              0             260
       polity              0             208
       single_basin      828             368
POS-04 buffer              0             260
       polity              0             208
       single_basin       19            1177
POS-05 buffer              0             260
       polity              0             208
       single_basin      158            1038
POS-06 buffer             49             211
       polity             46             162
       single_basin        0            1196
POS-07 buffer             15             245
   

In [7]:
# Cell 4 — Draft sentences on chosen fixtures (up to 2 per statement type), to read and judge
EXAMPLES = ['F01', 'F04', 'F18', 'F19', 'F22', 'F20', 'F32']
for qid in EXAMPLES:
    g = fdf[(fdf.qid == qid) & fdf.applies]
    print(f"--- {qid}: {FX_BY_ID[qid]['condition']}")
    for sid, gg in g.groupby('id'):
        for _, r in gg.head(2).iterrows():
            print(f"  {sid}  {r['text']}")
    print()


--- F01: arid; exogenous water (Niger); outside_active_domain
  POS-01  Aridity index (P/PET) ranks at the 11th percentile among all L6 basins worldwide.
  POS-01  Aridity index (P/PET) (upstream) ranks at the 36th percentile among all L6 basins worldwide.
  POS-03  Aridity index (P/PET) is 9 P/PET ×100.
  POS-03  Aridity index (P/PET) (upstream) is 47 P/PET ×100.
  POS-05  Cropland extent (upstream) is 9 %. Of all L6 basins, 41% are zero for this variable; ranked among the non-zero basins, this basin is at the 43rd percentile.
  POS-05  Flow distance to marine outlet is 2,331 km. Of all L6 basins, 33% are zero for this variable; ranked among the non-zero basins, this basin is at the 80th percentile.

--- F04: large buffer; suppressed scores; section 2.6 reference counts
  POS-02  Across the 9 basins in this buffer, Aridity index (P/PET) averages the 10th percentile among all L6 basins worldwide (area-weighted mean of the basins' percentiles).
  POS-02  Across the 9 basins in this buff

In [3]:
# Cell 5 — Population sample: how often each statement applies (share of all Band A-E rows), weighted to the world and not
pdf = A.apply_population(pop, W, R)
print('rows evaluated per statement type:', len(pdf) // len(R.STATEMENTS))
print('WEIGHTED by climate-zone size (percent of rows):')
print((A.rates(pdf) * 100).round(1).to_string())
print()
print('UNWEIGHTED (percent of rows):')
print((A.unweighted_rates(pdf) * 100).round(1).to_string())
print()
print('polities are not in the population sample (5 queries, all fixtures); see Cell 3 for polity counts')


rows evaluated per statement type: 26260
WEIGHTED by climate-zone size (percent of rows):
kind    buffer L6  single L6  single L8
id                                     
POS-01        0.1       52.3       53.3
POS-02       47.7        0.0        0.0
POS-03        0.2       66.1       64.4
POS-04        0.0        2.9        3.2
POS-05        0.0       11.0        7.9
POS-06       13.4        0.0        0.0
POS-07        5.8        0.0        0.0

UNWEIGHTED (percent of rows):
kind    buffer L6  single L6  single L8
id                                     
POS-01        0.5       47.3       47.8
POS-02       42.6        0.0        0.0
POS-03        0.8       61.6       60.2
POS-04        0.1        5.3        6.0
POS-05        0.1        9.0        6.3
POS-06       14.5        0.0        0.0
POS-07        5.7        0.0        0.0

polities are not in the population sample (5 queries, all fixtures); see Cell 3 for polity counts


In [8]:
# Cell 6 — Distribution of the number inside each statement, weighted (POS-01/05/02: percentile; POS-06: share of area with data at percentile 0)
print(A.stat_quantiles(pdf, ['POS-01', 'POS-02', 'POS-05', 'POS-06']).round(2).to_string(index=False))


    id      kind    n   p10   p25   p50   p75   p90
POS-01 buffer L6   28 15.47 22.05 67.64 90.00 97.55
POS-01 single L6 7522 12.69 29.18 52.17 77.19 90.54
POS-01 single L8 2488 14.43 29.99 54.89 77.11 90.83
POS-02 buffer L6 2191 11.88 27.84 51.07 72.57 86.62
POS-05 buffer L6    6 28.27 29.06 51.24 72.78 90.28
POS-05 single L6 1426 18.54 33.76 54.59 76.88 88.86
POS-05 single L8  330 17.77 23.66 52.72 72.09 85.32
POS-06 buffer L6  744  0.09  0.22  0.47  0.78  0.96


In [4]:
# Cell 7 — Sweep: POS-06 was drafted with "Most of" / "Part of" wording. Wording flips with the cutoff; evaluate a grid, do not choose
CUTS = [0.5, 0.6, 0.7, 0.8, 0.9]
sw = A.cutoff_sweep(pdf, 'POS-06', CUTS)
print('share of POS-06 rows that would read "Most of" at each cutoff, and share whose wording changes between 0.5 and 0.9:')
print(sw.round(3).to_string(index=False))
print()
print('variables whose wording flips (count of rows with a share from 0.5 up to 0.9):')
print(A.flipping_variables(pdf, 'POS-06', CUTS[0], CUTS[-1]).to_string())
print()
bs = pd.DataFrame(R.bottom_shares(pop), columns=['qid', 'scope', 'variable', 'adj'])
bs['w'] = bs.qid.map(W)
FLOORS = [0.005, 0.01, 0.05, 0.10, 0.25]
print('display floor: weighted share of candidate buffer rows that would get a POS-06 statement at each floor')
print(pd.DataFrame({f'>= {f}': [float((bs.w * (bs.adj >= f)).sum() / bs.w.sum())] for f in FLOORS}).round(3).to_string(index=False))


share of POS-06 rows that would read "Most of" at each cutoff, and share whose wording changes between 0.5 and 0.9:
     kind   n  >= 0.5  >= 0.6  >= 0.7  >= 0.8  >= 0.9  flips
buffer L6 744    0.48   0.433    0.34   0.222   0.111  0.369

variables whose wording flips (count of rows with a share from 0.5 up to 0.9):
variable
wet_pct_grp1                30
wet_pct_grp2_upstream       23
wet_pct_grp2                22
wet_pct_grp1_upstream       22
dist_sink                   20
karst_upstream              20
karst                       14
cropland_extent_upstream    13

display floor: weighted share of candidate buffer rows that would get a POS-06 statement at each floor
 >= 0.005  >= 0.01  >= 0.05  >= 0.1  >= 0.25
    0.229    0.219    0.207   0.194    0.154


In [9]:
# Cell 8 — Checks: what a raw 0 means, and rows where sea defeats the engine's outside_active_domain guard
print('(raw == 0, percentile == 0) over one-basin continuous ok rows:', dict(R.single_unit_zero_crosstab(pop + fx)))
d = pdf[pdf.applies & pdf.id.isin(['POS-01', 'POS-04', 'POS-05'])].groupby(['qid', 'variable']).size()
print('rows matching more than one of POS-01/04/05 (should be 0):', int((d > 1).sum()))
print()
sm = R.sea_guard_misses(pop + fx)
print(f'buffer rows where all the data-bearing area is at percentile 0 (share >= 0.90) but weight_at_zero < 0.90, so the guard did not fire: {len(sm)} rows in {len({x[0] for x in sm})} buffers')
per = pd.DataFrame(sm, columns=['qid', 'shortfall', 'variable', 'weight_at_zero']).groupby(['qid', 'shortfall']).size().rename('rows')
print(per.to_string())


(raw == 0, percentile == 0) over one-basin continuous ok rows: {(False, False): 12543, (True, True): 1188, (False, True): 3, (True, False): 63}
rows matching more than one of POS-01/04/05 (should be 0): 0

buffer rows where all the data-bearing area is at percentile 0 (share >= 0.90) but weight_at_zero < 0.90, so the guard did not fire: 119 rows in 19 buffers
qid    shortfall
F18    0.50          5
F19    0.75          6
P018b  0.19         11
P030b  0.22         11
P036b  0.12         11
P043b  0.34         11
P052b  0.03          1
P066b  0.37          9
P069b  0.95          6
P083b  0.40          8
P084b  0.14          8
P102b  0.10          4
P114b  0.36          4
P169b  0.51          6
P176b  0.38          3
P182b  0.43          5
P184b  0.32          5
P199b  0.22          4
P238b  0.22          1


In [5]:
# Cell 9 — Write this family's rows to the master table (replaces any earlier Position rows)
fxt, rw, ru = A.fixture_table(fdf), A.rates(pdf), A.unweighted_rates(pdf)
s0 = sw.iloc[0]
rows = []
for s in R.STATEMENTS:
    sid = s['id']
    f = fxt.loc[sid]
    row = dict(s, family='Position', applies_to=s['applies_to'], does_not_apply_when=s['does_not_apply_when'],
               in_ui_today=s['in_ui_today'], needs_new_data=s['needs_new_data'])
    row['fixture_result'] = f"{int(f.applied.sum())} rows applied, {int(f.not_applicable.sum())} not applicable ({len(fx)} fixtures)"
    row['population_result'] = ('share of all A-E rows, weighted: ' + ', '.join(f"{k} {rw.loc[sid, k]:.1%}" for k in rw.columns)
                                + '; unweighted: ' + ', '.join(f"{k} {ru.loc[sid, k]:.1%}" for k in ru.columns)
                                + '; polities not sampled')
    if sid == 'POS-06':
        row['sweep_result'] = (f"flips: 'Most of' wording changes for {s0['flips']:.0%} of applicable buffer rows between cutoffs 0.5 and 0.9 "
                               f"({s0['>= 0.5']:.0%} to {s0['>= 0.9']:.0%} would read 'Most of'); number alone adopted")
    else:
        row['sweep_result'] = 'no cutoff'
    rows.append(row)
n = P.upsert_master('Position', rows)
print('master table now has', n, 'rows ->', P.MASTER)


master table now has 13 rows -> /Users/karlg/Documents/repos/_edops/notebooks/edop/dashboard/wo02_master_table.csv
